# 0.1 — Global candidate coverage

This report answers one bounded question: how many candidate ions the local molecular databases provide under the explicit METASPACE-like polarity and mass-range conditions. It does not construct a dense deconvolution dictionary.

## Method

The configured precompute method `deconvolution.candidate_coverage` queries the materialized global catalogue once per condition and writes candidate-ion, compound, formula, and adduct counts. The candidate catalogue is built exclusively from `assets/local/metabolite_databases`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from msi_autoencoder_wrapper.analysis.precompute.cli import run_precompute_command
from msi_autoencoder_wrapper.visualization.metrics import plot_metric_tradeoff

REPOSITORY_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
NOTEBOOK_DIR = REPOSITORY_ROOT / 'assets/experiments/autoencoder_architecture/notebooks/annotation_model/19_09_deconvolution_inital'
SETTINGS_PATH = NOTEBOOK_DIR / 'analysis_settings.yaml'
RESULTS_DIR = NOTEBOOK_DIR / 'part_0_1_metaspace_candidate_coverage_results'
print(run_precompute_command(SETTINGS_PATH, background=False))

## Reproducible execution

Run the printed command once from the repository root. It executes the complete declared workflow: coverage, Torch numerical baseline, and small-dictionary identifiability. This notebook is intentionally read-only with respect to numerical artifacts.

In [ ]:
coverage_path = RESULTS_DIR / 'candidate_coverage.csv'
if not coverage_path.is_file():
    raise FileNotFoundError(f'Missing {coverage_path}. Run the command printed above.')
coverage = pd.read_csv(coverage_path)
coverage

In [ ]:
figure, axis = plot_metric_tradeoff(
    np.arange(len(coverage)),
    coverage['candidate_ion_count'].to_numpy(),
    x_label='Configured candidate-universe condition',
    metric='Candidate-ion count',
    label='local molecular databases',
)
axis.set_xticks(np.arange(len(coverage)), coverage['condition'], rotation=20, ha='right')
figure.savefig(RESULTS_DIR / 'candidate_coverage.png', bbox_inches='tight')
figure

## Interpretation

These counts define the global candidate population. The next two tests deliberately sample small, seeded subsets from this population before creating dense Torch matrices, avoiding a multi-gigabyte global `K` allocation.